# ④ 생성 — 질문에 답하기
담당: 생성 · 코드: `rag/generation/generation.py`

03 검증의 `artifacts/eval_set.jsonl` 질문을 재작성해 검색하고, 찾은 청크를 근거로 `gpt-5-mini` 가 원본 질문에 답합니다.
평가셋 순서대로 직전 질문·답변 최대 10개를 참고합니다. 사실의 근거는 현재 검색한 문서로 제한합니다.
답변 · 검색 근거 · 토큰 · 지연 시간과 `rewrite`(검색용 질문·모델·토큰·지연 시간)를 `artifacts/answers.jsonl` 로 저장합니다 → 05 평가가 읽습니다.
`.env` 에 Langfuse 키가 있으면 질문마다 Langfuse Cloud 에 추적 `answer-question`(재작성 `rewrite-question` + 검색 `retrieve-context` + 생성 `generate-answer` — 모델 · 토큰 · 비용)이 남고,
그 추적 id 가 `answers.jsonl` 의 `trace_id` 에 저장되어 05 평가가 점수를 붙입니다 (키가 없으면 기록하지 않고 똑같이 동작).

답을 하나 만들 때마다 `artifacts/checkpoints/answers/<설정 해시>.jsonl` 에 저장합니다. 중간에 끊기면 생성 셀을 다시 실행하세요 — 저장된 답으로 이전 대화를 되살려 다음 질문부터 이어서 만듭니다.
답변 · 검색 설정이나 청크가 바뀌면 새 파일로 처음부터, 03 에서 질문이 바뀌면 바뀐 질문부터 다시 만듭니다. 같은 설정으로 새로 답을 받으려면 `artifacts/checkpoints/answers/` 를 지우세요.

In [1]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.generation.generation import answer_questions
from rag.retrieval.retrieval import HybridIndex
from rag.usage import usage_summary

cfg = load_config()
retrieval_settings, = read_jsonl(ARTIFACTS / "retrieval_settings.jsonl")
cfg.update(retrieval_settings)


## 설정
답변 생성의 초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
rewrite 모델·추론 강도, 검색 비율과 리랭커 사용 여부는 02에서 저장한 `artifacts/retrieval_settings.jsonl`을 사용합니다. 먼저 02를 실행해야 하며, 검색 설정을 바꾸려면 02 설정 셀부터 다시 실행하세요.
답변 모델을 바꾸면 `LLM_PRICES`도 해당 모델의 입력 · 캐시 입력 · 출력 단가(USD / 100만 토큰)로 맞추세요. 생성 시 계산한 비용을 저장하며 05는 그 값을 사용합니다.


In [2]:
LLM_MODEL = "gpt-6-luna"                    # 예: "gpt-5-mini"
LLM_REASONING_EFFORT = cfg["llm"]["reasoning_effort"]  # 예: "low"
LLM_MAX_TOKENS = cfg["llm"]["max_tokens"]             # 출력·추론 토큰 한도
LLM_PRICES = cfg["llm"]["usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가

cfg["llm"].update(model=LLM_MODEL, reasoning_effort=LLM_REASONING_EFFORT, max_tokens=LLM_MAX_TOKENS,
                  usd_per_1m_tokens=LLM_PRICES)
cfg["llm"]


{'model': 'gpt-6-luna',
 'max_tokens': 2048,
 'reasoning_effort': 'low',
 'usd_per_1m_tokens': [0.25, 0.025, 2.0]}

In [3]:
eval_set = read_jsonl(ARTIFACTS / "eval_set.jsonl")
index = HybridIndex(ARTIFACTS, cfg)
answers = answer_questions(eval_set, index, cfg["llm"], cfg["rewrite"],
                           checkpoint_dir=ARTIFACTS / "checkpoints" / "answers")  # 끊기면 다시 실행해 이어서 생성
write_jsonl(answers, ARTIFACTS / "answers.jsonl")
pd.DataFrame(answers)[["id", "question", "answer", "sources", "latency_s"]]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 


저장된 답변 367/484개 — /home/spai1322/rfg_rag_retrieval/artifacts/checkpoints/answers/d2551f04c0ead499.jsonl


Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
Authentication error: 

생성 368/484 multi_hop-A-fill1-r1-0042 (6.36s)


...

Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
Authentication error: 

생성 483/484 needle-38 (5.03s)


Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Authentication error: Langfuse client initialized without public_key. Client will be disabled. Provide a public_key parameter or set LANGFUSE_PUBLIC_KEY environment variable. 
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.
Context error: No active span in current context. Operations that depend on an active span will be skipped. Ensure spans are created with start_as_current_observation() or that you're operating within an active span context.


생성 484/484 needle-39 (5.24s)


,id,question,answer,sources,latency_s
0,numeric-A-night1-r1-0003,한국철도공사의 「운행정보기록 자동분석시스」 사업에서 착수일로부터 사업기간은 얼마인가?,사업기간은 착수일로부터 180일입니다.[1],"[한국철도공사 (용역)_[재공고][긴급][협상형]운행정보기록 자동분석시스.hwp, ...",7.86
1,numeric-A-night1-r1-0004,한국농어촌공사의 「아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아」 ...,"용역금액은 **977,240,000원**입니다. [1]",[한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hw...,3.97
2,numeric-A-night1-r1-0021,을지대학교의 「을지대학교 비교과시스템 개발」 사업에서 교육정보전산시스템의 구축 년도는?,제공된 문서에서 확인할 수 없습니다. [1]에는 교육정보전산시스템을 활용해 통합 관...,"[을지대학교_을지대학교 비교과시스템 개발.hwp, 을지대학교_을지대학교 비교과시스템...",3.82
3,numeric-A-night1-r1-0023,한국원자력연구원의 「한국원자력연구원 선량평가시스템 고도화」 사업에서 사업 기간은 얼...,사업 기간은 **계약일로부터 6개월**입니다. [1],"[한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp, 한국원자력연구원_한...",2.50
4,numeric-A-night1-r1-0041,한국어촌어항공단의 「한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도」 사업에...,사업기간은 **계약일로부터 180일 이내(6개월)**입니다. [7],"[한국어촌어항공단_한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도.hwp, ...",5.38
...,...,...,...,...,...
479,needle-35,중앙선거관리위원회 「2025년도 행정정보시스템 위탁운영사업」의 PMR-011에 따라...,"PMR-011에 따르면, 사업종료 시한까지 중앙선거관리위원회의 산출물 제출 승인을 ...","[중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 중앙선거관리위원...",4.36
480,needle-36,한국발명진흥회의 「2024년 건설기술에 관한 특허·실용신안 활용실적 관리시스템 개편...,PMR-004는 **계약 후 10일 이내**에 **사업수행계획서(WBS 포함 의무사...,"[한국발명진흥회 입찰공고_2024년 건설기술에 관한 특허·실용신안 활용실.hwp, ...",4.96
481,needle-37,한국산업인력공단의 「RFID기반 국가자격 시험 결과물 스마트 관리시스템 도입 사업」...,"사업 완료 후 **14일이내** 사업 관련 자료를 제출해야 하며, 제출 비용은 **...","[한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp, ...",3.92
482,needle-38,한국수자원공사의 「건설통합시스템(CMS) 고도화」 PSR-001에서 무상 하자보수 ...,무상 하자보수 기간은 **K-water의 검사로 사업의 완성을 확인한 후 1년간**...,"[한국수자원공사_건설통합시스템(CMS) 고도화.hwp, 한국수자원공사_건설통합시스템...",5.03


## API 사용량
rewrite와 답변 생성 호출이 각각 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 두 표는 누적 사용량입니다.

In [4]:
usage_summary("rewrite")

,stage,evaluation,model,reasoning_effort,calls,input_tokens,cached_input_tokens,output_tokens,reasoning_tokens,cost_usd
0,rewrite,,gpt-6-luna,173,1,0,88,55,0.000219,
1,rewrite,,gpt-6-luna,174,1,0,98,63,0.000239,
2,rewrite,,gpt-6-luna,176,3,0,316,204,0.000764,
3,rewrite,,gpt-6-luna,177,2,0,228,151,0.000545,
4,rewrite,,gpt-6-luna,178,3,0,273,158,0.000679,
5,rewrite,,gpt-6-luna,179,5,0,618,420,0.001460,
6,rewrite,,gpt-6-luna,180,5,0,352,160,0.000929,
7,rewrite,,gpt-6-luna,181,2,0,173,90,0.000436,
8,rewrite,,gpt-6-luna,182,5,0,532,320,0.001292,
9,rewrite,,gpt-6-luna,183,5,0,476,270,0.001181,


In [5]:
usage_summary("answer")

,stage,evaluation,model,reasoning_effort,calls,input_tokens,cached_input_tokens,output_tokens,reasoning_tokens,cost_usd
0,answer,,gpt-6-luna,3218,1,0,124,97,0.001053,
1,answer,,gpt-6-luna,3263,1,0,87,0,0.000990,
2,answer,,gpt-6-luna,3281,1,0,49,0,0.000918,
3,answer,,gpt-6-luna,3300,1,0,25,0,0.000875,
4,answer,,gpt-6-luna,3306,1,0,66,0,0.000959,
...,...,...,...,...,...,...,...,...,...,...
261,answer,,gpt-6-luna,4087,1,0,78,58,0.001178,
262,answer,,gpt-6-luna,4091,1,0,21,0,0.001065,
263,answer,,gpt-6-luna,4096,1,0,17,0,0.001058,
264,answer,,gpt-6-luna,4241,1,0,36,0,0.001132,
